# Chapter 2: Model Training, Hyperparameter Tuning & Probabilistic Evaluation

This workbook implements the machine learning classification, hyperparameter optimization, and probabilistic calibration phase of the pipeline. In this notebook, we experiment with different hyperparameters and feature subsets, logging all trials and artifacts to a local MLflow server to systematically compare model performance. Using the Laplace-smoothed feature matrix generated during preprocessing, this notebook evaluates baseline linear classifiers (Logistic Regression) against non-linear gradient boosted decision trees (XGBoost) to predict an entity's geographical macro-region.

**Key Pipeline Stages:**
1. **Environment & Experiment Tracking:** Initializes analytical libraries and connects to a local MLflow tracking server for reproducible logging of parameters, metrics, and diagnostic plots.
2. **Probabilistic & Selective Classification Metrics:** Defines evaluation utilities beyond standard accuracy, including class-wise Calibration Curves, Brier Score Loss, and Area Under the Risk-Coverage Curve (AURC) to evaluate model reliability under selective abstention (Top 10%, 25%, and 50% confidence thresholds).
3. **Feature Space Preparation & Class Balancing:** Loads the engineered dataset, isolates behavioral and diurnal feature subsets, and computes balanced sample weights to counteract geographical class imbalance.
4. **Baseline Modeling (Logistic Regression):** Evaluates a linear baseline using 5-fold Stratified Cross-Validation.
5. **Bayesian Hyperparameter Tuning (XGBoost + Optuna):** Executes a 100-trial Optuna study to optimize tree architecture and regularization parameters. Uses a custom overfitting-penalized Brier score objective function: $\text{Score} = \text{Brier}_{\text{val}} + \lambda \vert{}\text{Brier}_{\text{val}} - \text{Brier}_{\text{train}}\vert{}$.

### 1: Environment Initialization & MLflow Tracking Setup

This section imports the core machine learning, optimization, and statistical libraries, and establishes a connection to the local MLflow server to systematically log all cross-validation experiments and visual artifacts.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import random
import mlflow

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score, f1_score, brier_score_loss, auc
from sklearn.preprocessing import StandardScaler, label_binarize, LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from mlflow.tracking import MlflowClient
from scipy.stats import circmean

import xgboost as xgb
from itertools import product

# Configure Pandas display settings to inspect full feature matrices
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 1000)

In [3]:
# Connect to the local MLflow tracking server and set the active experiment
# Note: Start server via CLI using: 
# mlflow server --backend-store-uri sqlite:///backend.db --default-artifact-root ./mlruns_artifacts
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("btc_timezone_analysis_model_training")


2026/10/23 07:24:41 INFO mlflow.tracking.fluent: Experiment with name 'btc_timezone_analysis_model_training' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/matej/mlruns_artifacts/1', creation_time=1792733081868, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792733081868, lifecycle_stage='active', name='btc_timezone_analysis_model_training', tags={}, trace_location=None, workspace='default'>

### 2: Diagnostic & Probabilistic Evaluation Functions

Because deanonymization requires high-confidence attribution, standard accuracy is insufficient. This section defines custom diagnostic functions to evaluate probabilistic calibration and selective classification performance:
* `class_report`: Generates side-by-side row-normalized confusion matrices for training and validation splits.   
* `calibration_curve_classes`: Evaluates whether predicted class probabilities reflect true empirical accuracy across quantile bins.   
* `calculate_aurc` & `plot_risk_coverage_curve`: Computes the Area Under the Risk-Coverage Curve (AURC), quantifying how the error rate (risk) decreases as the model abstains from low-confidence predictions.   
* `get_mlflow_confidence_metrics`: Aggregates fold-level Brier scores, overall accuracy, AURC, and selective accuracy at Top 10%, 25%, and 50% coverage thresholds. 

In [ ]:
def class_report(y_train, y_train_pred, y_val, y_val_pred, target_name):
    """Generates and saves normalized Confusion Matrices for Training and Validation sets."""
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 6), constrained_layout=True) 
    
    # 1. Training Confusion Matrix (Normalized by true row labels)
    ConfusionMatrixDisplay.from_predictions(
        y_train, y_train_pred, 
        ax=axes[0], cmap='Blues', normalize='true',
        labels=target_name, values_format='.2f',
        colorbar=False  
    )
    axes[0].set_title("Training Confusion Matrix (Normalized)", fontsize=16, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].tick_params(axis='y', rotation=45)
    axes[0].set_xlabel("Predicted label", fontsize=14)
    axes[0].set_ylabel("True label", fontsize=14)
    
    # 2. Validation Confusion Matrix (Normalized by true row labels)
    ConfusionMatrixDisplay.from_predictions(
        y_val, y_val_pred, 
        ax=axes[1], cmap='Greens', normalize='true',
        labels=target_name, values_format='.2f',
        colorbar=False  
    )
    axes[1].set_title("Validation Confusion Matrix (Normalized)", fontsize=16, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].tick_params(axis='y', rotation=45) 
    axes[1].set_xlabel("Predicted label", fontsize=14)
    axes[1].set_ylabel("True label", fontsize=14)
    
    # Export high-resolution figure for MLflow artifact logging
    plt.savefig("./plots/class_report.png", format='png', dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
def calibration_curve_classes(y_train, y_train_proba, y_train_pred, y_val, y_val_proba, y_val_pred, colors, labels):
    """Generates per-region reliability diagrams (Calibration Curves) for Train and Validation sets."""
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    
    for j, a in zip(range(2), ["Train", "Val"]):
        # Plot ideal diagonal calibration reference line and 0.75 confidence threshold
        ax[j].plot([0, 1], [0, 1], "k--", label="Perfectly Calibrated")
        ax[j].plot([0.75, 0.75], [-0.05, 1.05], "--", color="#C0C0C0")
        
        for l, c in zip(labels, colors):
            proba = locals()[f"y_{a.lower()}_proba"]
            pred  = locals()[f"y_{a.lower()}_pred"]
            true  = locals()[f"y_{a.lower()}"]
        
            # Isolate samples belonging to the current true macro-region
            subset = (true == l)
            
            confidences = np.max(proba[subset], axis=1)
            accuracies = (pred[subset] == true[subset]).astype(int)
            
            # Partition predictions into 5 quantile-based confidence bins
            n_bins = 5
            percentiles = np.linspace(0.0, 1.0, n_bins + 1)
            bins = np.quantile(confidences, percentiles)
            bin_indices = np.digitize(confidences, bins) - 1
            bin_indices[bin_indices == n_bins] = n_bins - 1
            
            bin_accuracies = []
            bin_confidences = []
            bin_percentages = []
            
            # Compute empirical accuracy vs. mean predicted confidence per bin
            for i in range(n_bins):
                mask = (bin_indices == i)
                if np.sum(mask) > 0: # Ensure the bin is non-empty
                    bin_accuracies.append(np.mean(accuracies[mask]))
                    bin_confidences.append(np.mean(confidences[mask]))
                    bin_percentages.append((np.sum(mask) / len(confidences)) * 100)
        
            ax[j].plot(bin_confidences, bin_accuracies, marker='o', linestyle='-', color=c, label=l)
        
        ax[j].set_xlabel("Confidence", fontsize=14)
        ax[j].set_ylabel("True Accuracy", fontsize=14)
        ax[j].set_title(f"Calibration Graph ({a} Set)", fontsize=16, fontweight='bold')
        ax[j].legend(loc="upper left")
        ax[j].grid(True, alpha=0.5)
        ax[j].set_xlim([-0.05, 1.05])
        ax[j].set_ylim([-0.05, 1.05])
        
    plt.savefig("./plots/calibration_graph.png", dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
def calculate_aurc(y_true, y_prob, y_pred):
    """
    Calculates the Area Under the Risk-Coverage Curve (AURC) to evaluate 
    selective classification performance.
    """
    # 1. Convert to NumPy arrays to prevent Pandas index alignment issues
    y_true_np = np.array(y_true)
    y_pred_np = np.array(y_pred)
    
    # 2. Extract maximum predicted class probability as model confidence
    confidences = np.max(y_prob, axis=1)
    
    # 3. Compute binary error indicator (1 = Misclassified, 0 = Correct)
    errors = (y_true_np != y_pred_np).astype(int)
    
    # 4. Rank samples by descending model confidence
    sorted_indices = np.argsort(-confidences)
    sorted_errors = errors[sorted_indices]
    
    # 5. Compute Coverage (proportion of dataset retained)
    n_samples = len(y_true_np)
    coverages = np.arange(1, n_samples + 1) / n_samples
    
    # 6. Compute cumulative Risk (selective error rate at each coverage level)
    cumulative_errors = np.cumsum(sorted_errors)
    risks = cumulative_errors / np.arange(1, n_samples + 1)
    
    # 7. Anchor curve at 0.0 coverage
    coverages = np.insert(coverages, 0, 0.0)
    risks = np.insert(risks, 0, risks[0]) 
    
    # 8. Integrate area under the curve via trapezoidal rule
    aurc_score = auc(coverages, risks)
    
    return coverages, risks, aurc_score

In [ ]:
def get_mlflow_confidence_metrics(
    y_val_true, y_val_pred, y_val_prob, 
    y_train_true, y_train_pred, y_train_prob, 
    target_names
):
    """Computes global and fold-averaged calibration, accuracy, and coverage metrics for MLflow."""
    metrics = {}

    # 1. Global Brier Scores (probabilistic MSE) and Overall Accuracy
    metrics['train_brier_score'] = float(brier_score_loss(label_binarize(y_train_true, classes=target_names), y_train_prob))
    metrics['val_brier_score'] = float(brier_score_loss(label_binarize(y_val_true, classes=target_names), y_val_prob))
    metrics['train_accuracy'] = float(accuracy_score(y_train_true, y_train_pred))
    metrics['val_accuracy'] = float(accuracy_score(y_val_true, y_val_pred))
    metrics['samples'] = len(y_val_true)

    fold_metrics_collection = {
        "val_coverage_10_accuracy": [], 
        "val_coverage_25_accuracy": [], 
        "val_coverage_50_accuracy": [], 
        "val_aurc": [],
        "train_aurc": []
    }
    
    n_splits = 5
    
    # Compute fold slice sizes (train array is 4x larger than validation in 5-fold CV)
    val_fold_size = len(y_val_true) // n_splits
    train_fold_size = len(y_train_true) // n_splits
    
    for i in range(n_splits):
        # 1. Extract Validation Fold Slice
        val_start = i * val_fold_size
        val_end = (i + 1) * val_fold_size if i < n_splits - 1 else len(y_val_true)
        
        y_val_true_fold = np.array(y_val_true[val_start:val_end])
        y_val_pred_fold = np.array(y_val_pred[val_start:val_end])
        y_val_prob_fold = np.array(y_val_prob[val_start:val_end])
        
        # 2. Extract Training Fold Slice
        train_start = i * train_fold_size
        train_end = (i + 1) * train_fold_size if i < n_splits - 1 else len(y_train_true)
        
        y_train_true_fold = np.array(y_train_true[train_start:train_end])
        y_train_pred_fold = np.array(y_train_pred[train_start:train_end])
        y_train_prob_fold = np.array(y_train_prob[train_start:train_end])

        # 3. Compute Selective Accuracy at Top 10%, 25%, and 50% Confidence Thresholds
        confidences = np.max(y_val_prob_fold, axis=1)
        sorted_idx = np.argsort(confidences)[::-1]
        y_true_sorted = y_val_true_fold[sorted_idx]
        y_pred_sorted = y_val_pred_fold[sorted_idx]
        
        coverages = [0.10, 0.25, 0.50]
        for c in coverages:
            k = max(1, int(c * len(y_val_true_fold))) 
            top_true = y_true_sorted[:k]
            top_pred = y_pred_sorted[:k]
            
            c_str = f"val_coverage_{int(c * 100)}"
            fold_metrics_collection[f"{c_str}_accuracy"].append(float(accuracy_score(top_true, top_pred)))
            
        # 4. Compute Fold-level AURC for Validation and Training sets
        _, _, val_fold_aurc = calculate_aurc(y_val_true_fold, y_val_prob_fold, y_val_pred_fold)
        fold_metrics_collection['val_aurc'].append(float(val_fold_aurc))
        
        _, _, train_fold_aurc = calculate_aurc(y_train_true_fold, y_train_prob_fold, y_train_pred_fold)
        fold_metrics_collection['train_aurc'].append(float(train_fold_aurc))
        
    # Average selective metrics across all 5 cross-validation folds
    for key, values in fold_metrics_collection.items():
        metrics[key] = float(np.mean(values))
    
    return metrics

In [ ]:
def plot_risk_coverage_curve(y_train, y_train_proba, y_train_pred, y_val, y_val_proba, y_val_pred):
    """Plots side-by-side Risk-Coverage curves for Training and Validation sets."""
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    datasets = [
        ("Train", y_train, y_train_proba, y_train_pred, axes[0]),
        ("Val", y_val, y_val_proba, y_val_pred, axes[1])
    ]
    
    for name, true, proba, pred, ax in datasets:
        coverages, risks, aurc_score = calculate_aurc(true, proba, pred)
        
        # Plot selective risk trajectory
        ax.plot(coverages, risks, color='blue', linewidth=2, label=f'Model AURC={aurc_score:.4f}')
        
        # Plot full-dataset baseline error rate (100% coverage)
        baseline_error = np.mean(np.array(true) != np.array(pred))
        ax.axhline(y=baseline_error, color='red', linestyle='--', label=f'Full Dataset Error ({baseline_error:.4f})')
        
        ax.set_title(f"Risk-Coverage Curve ({name} Set)", fontsize=16, fontweight='bold')
        ax.set_xlabel("Coverage (Proportion of dataset evaluated)", fontsize=14)
        ax.set_ylabel("Risk (Error rate on covered samples)", fontsize=14)
        ax.legend(loc="upper left")
        ax.grid(True, alpha=0.5)
        ax.set_xlim([-0.05, 1.05])
        
    plt.savefig("./plots/risk_coverage.png", dpi=300, bbox_inches="tight")
    plt.close()

### 3: Dataset Loading & Feature Space Selection

Loads the preprocessed entity dataset and defines the active feature columns (col_names), allowing iterative evaluation of the 24-hour baseline vector against extended behavioral features (weekday/weekend splits, total volume, lifespan, 5-hour resting ratios, and DST shift).

In [ ]:
# Load the preprocessed and Laplace-smoothed dataset
data = pd.read_csv("/home/matej/btc_timezone_analysis/02_data_preprocessing/database_from_bitcointalk.csv")
len(data)

1934

In [ ]:
# Ensure all column headers are cast to string format
data.columns = data.columns.astype(str)
data.head()

,Unnamed: 0,entity,offset,is_regional_location,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,total,log_total,min_5h_activity,min_5h_activity_sin,min_5h_activity_cos,min_5h_activity_%,lifespan_days,0_weekday,1_weekday,2_weekday,3_weekday,4_weekday,5_weekday,6_weekday,7_weekday,8_weekday,9_weekday,10_weekday,11_weekday,12_weekday,13_weekday,14_weekday,15_weekday,16_weekday,17_weekday,18_weekday,19_weekday,20_weekday,21_weekday,22_weekday,23_weekday,0_weekend,1_weekend,2_weekend,3_weekend,4_weekend,5_weekend,6_weekend,7_weekend,8_weekend,9_weekend,10_weekend,11_weekend,12_weekend,13_weekend,14_weekend,15_weekend,16_weekend,17_weekend,18_weekend,19_weekend,20_weekend,21_weekend,22_weekend,23_weekend,total_weekday,total_weekend,dst_shift,region
0,0,Entity 100,7.0,0,0.020921,0.037657,0.025105,0.050209,0.050209,0.058577,0.033473,0.054393,0.050209,0.046025,0.046025,0.071130,0.037657,0.033473,0.054393,0.054393,0.062762,0.112971,0.037657,0.016736,0.012552,0.016736,0.004184,0.012552,215,5.370638,21,-0.707107,0.707107,0.062762,527.561829,0.022599,0.050847,0.028249,0.050847,0.067797,0.073446,0.039548,0.045198,0.033898,0.056497,0.050847,0.050847,0.028249,0.022599,0.056497,0.073446,0.084746,0.062147,0.033898,0.022599,0.011299,0.016949,0.005650,0.011299,0.023256,0.011628,0.023256,0.046512,0.011628,0.023256,0.023256,0.069767,0.081395,0.023256,0.034884,0.104651,0.058140,0.058140,0.046512,0.011628,0.011628,0.197674,0.046512,0.011628,0.023256,0.023256,0.011628,0.023256,153.0,62.0,0.0,East_Asia_Pac
1,1,Entity 1000,2.0,0,0.107143,0.107143,0.071429,0.035714,0.017857,0.035714,0.053571,0.017857,0.035714,0.035714,0.089286,0.035714,0.053571,0.053571,0.017857,0.017857,0.017857,0.035714,0.017857,0.035714,0.017857,0.035714,0.017857,0.035714,32,3.465736,16,-0.866025,-0.500000,0.107143,125.700324,0.100000,0.100000,0.080000,0.040000,0.020000,0.020000,0.040000,0.020000,0.040000,0.040000,0.080000,0.040000,0.060000,0.040000,0.020000,0.020000,0.020000,0.040000,0.020000,0.040000,0.020000,0.040000,0.020000,0.040000,0.066667,0.066667,0.033333,0.033333,0.033333,0.066667,0.066667,0.033333,0.033333,0.033333,0.066667,0.033333,0.033333,0.066667,0.033333,0.033333,0.033333,0.033333,0.033333,0.033333,0.033333,0.033333,0.033333,0.033333,26.0,6.0,11.0,Euro_Africa
2,2,Entity 1001,2.0,0,0.034483,0.034483,0.051724,0.051724,0.068966,0.068966,0.103448,0.172414,0.051724,0.034483,0.034483,0.034483,0.017241,0.051724,0.017241,0.017241,0.017241,0.034483,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,34,3.526361,20,-0.866025,0.500000,0.086207,1093.023981,0.035714,0.035714,0.035714,0.053571,0.071429,0.071429,0.107143,0.178571,0.053571,0.035714,0.035714,0.035714,0.017857,0.053571,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.038462,0.038462,0.076923,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.076923,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,32.0,2.0,-4.0,Euro_Africa
3,3,Entity 1002,-5.0,0,0.053571,0.071429,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.035714,0.035714,0.053571,0.071429,0.125000,0.035714,0.053571,0.017857,0.071429,0.053571,0.035714,0.107143,32,3.465736,4,0.866025,0.500000,0.089286,1229.220995,0.066667,0.066667,0.022222,0.022222,0.022222,0.022222,0.022222,0.022222,0.022222,0.022222,0.022222,0.022222,0.022222,0.022222,0.044444,0.088889,0.133333,0.044444,0.066667,0.022222,0.066667,0.066667,0.022222,0.044444,0.028571,0.057143,0.028571,0.028571,0.028571,0.028571,0.028571,0.028571,0.028571,0.028571,0.028571,0.028571,0.057143,0.057143,0.057143,0.028571,0.057143,0.028571,0.028571,0.028571,0.057143,0.028571,0.057143,0.142857,21.0,11.0,2.0,America
4,5,Entity 1004,7.0,0,0.017241,0.017241,0.017241,0.051724,0.086207,0.068966,0.086207,0.051724,0.017241,0.034483,0.086207,0.017241,0.103448,0.051724,0.017241,0.017241,0.051724,0.034483,0.068966,0.017241,0.034483,0.017241,0.017241,0.01

In [ ]:
# Inspect available feature columns
data.columns

Index(['Unnamed: 0', 'entity', 'offset', 'is_regional_location', '0', '1', '2',
       '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15',
       '16', '17', '18', '19', '20', '21', '22', '23', 'total', 'log_total',
       'min_5h_activity', 'min_5h_activity_sin', 'min_5h_activity_cos',
       'min_5h_activity_%', 'lifespan_days', '0_weekday', '1_weekday',
       '2_weekday', '3_weekday', '4_weekday', '5_weekday', '6_weekday',
       '7_weekday', '8_weekday', '9_weekday', '10_weekday', '11_weekday',
       '12_weekday', '13_weekday', '14_weekday', '15_weekday', '16_weekday',
       '17_weekday', '18_weekday', '19_weekday', '20_weekday', '21_weekday',
       '22_weekday', '23_weekday', '0_weekend', '1_weekend', '2_weekend',
       '3_weekend', '4_weekend', '5_weekend', '6_weekend', '7_weekend',
       '8_weekend', '9_weekend', '10_weekend', '11_weekend', '12_weekend',
       '13_weekend', '14_weekend', '15_weekend', '16_weekend', '17_weekend',
       '18_weekend', '19

In [ ]:
# Define target variable (macro-region) and select active feature space
X = data
y = data["region"]

# Full feature bundle: 24h base profile + weekday/weekend profiles + behavioral metadata
col_names = [
    "0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "20", "21", "22", "23",
    "0_weekday", "1_weekday", "2_weekday", "3_weekday", "4_weekday", "5_weekday", "6_weekday", "7_weekday", "8_weekday", "9_weekday", "10_weekday", "11_weekday", "12_weekday", "13_weekday", "14_weekday", "15_weekday", "16_weekday", "17_weekday", "18_weekday", "19_weekday", "20_weekday", "21_weekday", "22_weekday", "23_weekday",
    "0_weekend", "1_weekend", "2_weekend", "3_weekend", "4_weekend", "5_weekend", "6_weekend", "7_weekend", "8_weekend", "9_weekend", "10_weekend", "11_weekend", "12_weekend", "13_weekend", "14_weekend", "15_weekend", "16_weekend", "17_weekend", "18_weekend", "19_weekend", "20_weekend", "21_weekend", "22_weekend", "23_weekend",
    "total", "min_5h_activity", "min_5h_activity_%", "lifespan_days", "dst_shift"
]

# Alternative configuration: 24h base profile + DST shift only
# col_names = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "20", "21", "22", "23", "dst_shift"]

### 4: Baseline Model – Multinomial Logistic Regression

Before training complex non-linear ensembles, we establish a linear baseline using Multinomial Logistic Regression wrapped in a StandardScaler pipeline. Class imbalance is addressed via inverse-frequency class weighting (class_weight='balanced'), evaluated across a 5-fold Stratified Cross-Validation split.

In [ ]:
# Perform stratified train-test split (75% Training/CV, 25% Holdout Test)
num = 1019
print(num)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=num)

# Compute balanced sample weights inversely proportional to class frequencies
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

1019


In [ ]:
target_names = ["America", "Euro_Africa", "Central_South_Asia", "East_Asia_Pac"]
colors = ['red', 'green', 'blue', 'black']

# Execute 5-fold Stratified Cross-Validation and log results to MLflow
with mlflow.start_run(run_name="logistic_regression_model"):
    mlflow.set_tag("model_type", "logistic regression test")
    mlflow.set_tag("data type", "24 hour transaction distribution")
    mlflow.log_param("random_state train - test split", num)
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('log_reg', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ])

    kf = StratifiedKFold(n_splits=5, shuffle=True)

    y_val_cv, y_val_cv_pred, y_val_cv_proba = [], [], []
    y_train_cv, y_train_cv_pred, y_train_cv_proba = [], [], []

    for train_idx, val_idx in kf.split(X_train[col_names], y_train):
        X_tr, X_va = X_train[col_names].iloc[train_idx], X_train[col_names].iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # Fit standardized logistic regression model
        pipeline.fit(X_tr, y_tr)

        # Collect Training fold predictions and probabilities
        y_train_cv.extend(y_tr)
        y_train_cv_pred.extend(pipeline.predict(X_tr))
        y_train_cv_proba.extend(pipeline.predict_proba(X_tr))
        
        # Collect Validation fold predictions and probabilities
        y_val_cv.extend(y_va)
        y_val_cv_pred.extend(pipeline.predict(X_va))
        y_val_cv_proba.extend(pipeline.predict_proba(X_va))
    
    # Convert accumulated predictions to NumPy arrays
    y_train_cv = np.array(y_train_cv)
    y_train_cv_pred = np.array(y_train_cv_pred)
    y_train_cv_proba = np.array(y_train_cv_proba)
    y_val_cv = np.array(y_val_cv)
    y_val_cv_pred = np.array(y_val_cv_pred)
    y_val_cv_proba = np.array(y_val_cv_proba)

    # Calculate and log probabilistic and selective coverage metrics
    metrics = get_mlflow_confidence_metrics(
        y_val_cv, y_val_cv_pred, y_val_cv_proba, 
        y_train_cv, y_train_cv_pred, y_train_cv_proba, 
        pipeline.classes_
    )
    mlflow.log_metrics(metrics)
    
    # Generate and log diagnostic plots
    class_report(y_train_cv, y_train_cv_pred, y_val_cv, y_val_cv_pred, target_names)
    calibration_curve_classes(y_train_cv, y_train_cv_proba, y_train_cv_pred, y_val_cv, y_val_cv_proba, y_val_cv_pred, colors, target_names)
    plot_risk_coverage_curve(y_train_cv, y_train_cv_proba, y_train_cv_pred, y_val_cv, y_val_cv_proba, y_val_cv_pred)
    
    mlflow.log_artifact("./plots/class_report.png", artifact_path="plots")
    mlflow.log_artifact("./plots/calibration_graph.png", artifact_path="plots")
    mlflow.log_artifact("./plots/risk_coverage.png", artifact_path="plots")

### 5: XGBoost Classifier & Bayesian Hyperparameter Tuning (Optuna)

To capture non-linear interactions across the 24-hour activity bins while preventing overfitting on noisy blockchain data, we tune an `XGBClassifier` using Optuna.   

Rather than optimizing raw validation accuracy, the objective function minimizes a **Penalized Brier Score**:
$$
\text{Objective} = \text{Brier}_{\text{val}} + \lambda \cdot \vert{}\text{Brier}_{\text{val}} - \text{Brier}_{\text{train}}\vert{}
$$
where we experimented with different penalty weights $\lambda \in \{0.5, 0.75, 1.0, 2.5, 5.0\}$ to explicitly penalize divergence between training and validation calibration.

In [ ]:
# Define hyperparameter search bounds for Optuna Bayesian optimization
search_space = {
    "n_estimators": [100, 1000],
    "eta": [0.01, 1.0],          # Learning rate (sampled in log-space)
    "gamma": [0.0, 20.0],        # Minimum loss reduction required for a split
    "reg_lambda": [0.0, 10.0],   # L2 regularization weight
    "max_depth": [2, 8],         # Maximum tree depth
    "min_child_weight": [1, 20], # Minimum sum of instance weight needed in a child
    "subsample": [0.6, 1.0],     # Row subsampling ratio per tree
    "colsample_bytree": [0.6, 1.0] # Feature subsampling ratio per tree
}

In [ ]:
def objective(trial):
    """Optuna objective function for 5-fold cross-validated XGBoost tuning."""
    with mlflow.start_run(nested=True, run_name=f"XGBoost_{trial.number}"):
        target_names = ["America", "Euro_Africa", "Central_South_Asia", "East_Asia_Pac"]
        colors = ['red', 'green', 'blue', 'black']
        
        # Overfitting penalty weight (lambda)
        lambda_par = 0.5
        experiment_name = f"xgboost hyperparameter fitting with lambda={lambda_par} with higher min eta and higher max gamma"
        mlflow.set_tag("model_type", experiment_name)
        mlflow.set_tag("data type", "24 hour transaction distribution + all additional")
        
        # Log hyperparameter search space ranges as MLflow tags
        range_tags = {f"range_{k}": str(v) for k, v in search_space.items()}
        mlflow.set_tags(range_tags)
        
        # Sample hyperparameters for the current trial
        param = {
            "n_estimators": trial.suggest_int("n_estimators", search_space["n_estimators"][0], search_space["n_estimators"][1]),
            "eta": trial.suggest_float("eta", search_space["eta"][0], search_space["eta"][1], log=True),
            "gamma": trial.suggest_float("gamma", search_space["gamma"][0], search_space["gamma"][1]),
            "reg_lambda": trial.suggest_float("reg_lambda", search_space["reg_lambda"][0], search_space["reg_lambda"][1]),
            "max_depth": trial.suggest_int("max_depth", search_space["max_depth"][0], search_space["max_depth"][1]),
            "min_child_weight": trial.suggest_int("min_child_weight", search_space["min_child_weight"][0], search_space["min_child_weight"][1]),
            "subsample": trial.suggest_float("subsample", search_space["subsample"][0], search_space["subsample"][1]),
            "colsample_bytree": trial.suggest_float("colsample_bytree", search_space["colsample_bytree"][0], search_space["colsample_bytree"][1]),
        }

        kf = StratifiedKFold(n_splits=5, shuffle=True)

        y_val_cv, y_val_cv_pred, y_val_cv_proba = [], [], []
        y_train_cv, y_train_cv_pred, y_train_cv_proba = [], [], []

        # Encode string region labels to integers for XGBoost compatibility
        le = LabelEncoder()
        y_train_encoded = le.fit_transform(y_train)
        pipeline = xgb.XGBClassifier(**param)

        for train_idx, val_idx in kf.split(X_train[col_names], y_train_encoded):
            X_tr, X_va = X_train[col_names].iloc[train_idx], X_train[col_names].iloc[val_idx]
            y_tr, y_va = y_train_encoded[train_idx], y_train_encoded[val_idx]
            
            # Compute fold-specific balanced sample weights and fit model
            sample_weights = compute_sample_weight(class_weight='balanced', y=y_tr)
            pipeline.fit(X_tr, y_tr, sample_weight=sample_weights)

            # Store inverse-transformed Training predictions and probabilities
            y_train_cv.extend(le.inverse_transform(y_tr))
            y_train_cv_pred.extend(le.inverse_transform(pipeline.predict(X_tr)))
            y_train_cv_proba.extend(pipeline.predict_proba(X_tr))
            
            # Store inverse-transformed Validation predictions and probabilities
            y_val_cv.extend(le.inverse_transform(y_va))
            y_val_cv_pred.extend(le.inverse_transform(pipeline.predict(X_va)))
            y_val_cv_proba.extend(pipeline.predict_proba(X_va))

        # Convert lists to NumPy arrays
        y_train_cv = np.array(y_train_cv)
        y_train_cv_pred = np.array(y_train_cv_pred)
        y_train_cv_proba = np.array(y_train_cv_proba)
        y_val_cv = np.array(y_val_cv)
        y_val_cv_pred = np.array(y_val_cv_pred)
        y_val_cv_proba = np.array(y_val_cv_proba)

        # Evaluate and log metrics and diagnostic plots
        metrics = get_mlflow_confidence_metrics(
            y_val_cv, y_val_cv_pred, y_val_cv_proba, 
            y_train_cv, y_train_cv_pred, y_train_cv_proba, 
            le.inverse_transform(pipeline.classes_)
        )
        mlflow.log_params(param)
        mlflow.log_param("lambda", lambda_par)
        mlflow.log_param("random_state train - test split", num)
        mlflow.log_metrics(metrics)
        
        class_report(y_train_cv, y_train_cv_pred, y_val_cv, y_val_cv_pred, target_names)
        calibration_curve_classes(y_train_cv, y_train_cv_proba, y_train_cv_pred, y_val_cv, y_val_cv_proba, y_val_cv_pred, colors, target_names)
        plot_risk_coverage_curve(y_train_cv, y_train_cv_proba, y_train_cv_pred, y_val_cv, y_val_cv_proba, y_val_cv_pred)
        
        mlflow.log_artifact("./plots/class_report.png", artifact_path="plots")
        mlflow.log_artifact("./plots/calibration_graph.png", artifact_path="plots")
        mlflow.log_artifact("./plots/risk_coverage.png", artifact_path="plots")
            
        # Compute Penalized Brier Score to penalize train-validation divergence
        final_score = metrics["val_brier_score"] + lambda_par * np.abs(metrics["val_brier_score"] - metrics["train_brier_score"])
        mlflow.log_metric("optuna_penalized_score", final_score)
        
    return final_score

In [ ]:
# Re-initialize the stratified train-test split and sample weights prior to study execution
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=num)
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

In [ ]:
# Initialize and execute the Optuna study (100 trials, minimizing penalized Brier score)
study = optuna.create_study(study_name="xgboost_study", direction="minimize")
study.optimize(objective, n_trials=100, show_progress_bar=False, n_jobs=1)